<h1 style="font-size: 40px; margin-bottom: 0px;">5.1 Python image analysis (I)</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 800px;"></hr>

We're all familiar with using ImageJ for analyzing images, but here, we'll explore how a computer understands images and image files and then use Python to perform many of the same analyses that we performed using ImageJ in MCB201A with the goal of working towards automating an image analysis workflow. 

For today's notebook, we'll explore how images are represented on our computers in order to learn process our images. To do this, we'll make use of a package called <code>scikit-image</code>, which is an image processing package that uses numpy arrays. <a href="https://scikit-image.org/" rel="noopener noreferrer"><u>Information, including documentation, on <code>scikit-image</code> can be found here.</u></a> We'll then make use of our understanding of conditional statements and <code>scikit-image</code> to segment our images in preparation for downstream quantification.

<strong>Learning objectives:</strong>
<ul>
    <li>Learn how to import images in a Python notebook</li>
    <li>Understand how images are represented</li>
    <li>Learn how to display images in a Python notebook</li>
    <li>Learn how to process images</li>
</ul>

<h1 style="font-size: 40px; margin-bottom: 0px;">Import packages for today</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 600px;"></hr>

Today, we'll make use of our usual packages:

<ul>
    <li><code>numpy</code></li>
    <li><code>pandas</code></li>
    <li><code>matplotlib.pyplot</code></li>
    <li><code>seaborn</code></li>
    <li><code>os</code></li>
</ul>

And we'll make use of two packages that we haven't used before:

<ul>
    <li><code>skimage</code></li>
    <li><code>scipy.ndimage</code></li>
</ul>

These two packages are useful for processing images in Python to get them ready for analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import skimage as ski
import scipy.ndimage as ndi

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #1: What is an image?</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 600px;"></hr>

For this exercise, you'll play around with an image in your notebook to better understand how an image is represented on a computer so that you can then understand how we can make use of this representation to extract quantitative information from our images.

Images on a computer are essentially just a 2D matrix of values, ranging from 0 to 255 (for 8-bit images), and each value indicates the intensity of a single pixel. For a simple grayscale image, 0 represents pure black and 255 represents pure white, and all the values between 0 and 255 represent the shades of gray between pure black to pure white. 

<h2>A little bit about bits</h2>

To understand why the values for an 8-bit image range from 0 to 255, you'll need to understand bits and bit-depth. Each bit is a basic unit in computing (basically on or off), and each bit can only be represented by one of two possible values, either 0 or 1. In the example tables below, each individual bit is a white square within the table, and each row depicts the possible combination of bits given the bit-depth, and the value is shown next to it.

<table>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">1-bit example</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Value</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">0</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 2</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
            <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
            <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">1</td>
    </tr>
</table>

&nbsp;

<table>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">2-bit example</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Value</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">0</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 2</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">1</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 3</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">2</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 4</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">3</td>
    </tr>
</table>

&nbsp;

<table>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">4-bit example</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Value</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">0</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 2</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">1</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 3</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">2</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 4</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">3</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 5</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">4</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 6</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">5</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 7</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">6</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 8</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">7</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 9</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">8</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 10</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">9</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 11</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">10</td>
    </tr>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 12</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">11</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 13</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">12</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 14</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">13</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 15</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">0</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">14</td>
    </tr>
    <tr style="background-color: transparent; border: 0; border-color: #000000;">
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">Possibility 16</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 1px solid; border-color: #000000;">1</td>
        <td style="background-color: transparent; border: 0; border-color: #000000;">&nbsp;</td>
        <td style="background-color: #EEEEEE; border: 1px solid; border-color: #000000;">15</td>
    </tr>
</table>


Therefore, if there are 8 bits, then there are <strong>2<sup>8</sup></strong> possible combinations of bits, which gives us a value space of 256 possible values. Since the first value is zero, the range of values is then 0 to 255. If we instead have a 16-bit or 32-bit image, we then increase that value space to <strong>2<sup>16</sup></strong> and <strong>2<sup>32</sup></strong> respectively, meaning that a 16-bit image has a larger value space than an 8-bit image, but smaller value space than a 32-bit image (HDR image).

For images, the bit-depth then provides us with the value space available to represent intensities of light as a numerical value on a computer. Something to keep in mind is that in reality, intensity of light exists on a continuum, whereas we are limited to representing that intensity in discrete values on a computer, where the possible values are determined by the bit-depth. And this has practical implications as well in terms of imaging, image processing, rendering, diagnostics, and analysis.

<p style="font-size: 16px; text-align: center;"><strong>Image bit-depth</strong></p>
<img src="./ref-images/bit-depth.png" style="height: 300px; margin: auto"/>

You can use the code cell below to play around with how much a 8-bit vs 16-bit vs 32-bit value space will differ by looking at the difference in <code>2&ast;&ast;8</code> vs <code>2&ast;&ast;16</code> vs <code>2&ast;&ast;32</code>.

<h2>Explore a simple image</h2>

First, set up a 2D array of zeros using <code>np.zeros()</code>. Recall that we've used this function to set up a 1D array of zeros for holding the outputs from our computational models. If you dig into <a href="https://numpy.org/doc/stable/reference/generated/numpy.zeros.html" rel="noopener noreferrer"><u>the documentation for <code>np.zeros()</code>,</u></a> you'll see that we can also specify a 2D array as well by simply passing a tuple containing the number of rows and columns instead of a single integer object.

Use the code cell below to set up a 10x10 matrix of zeros.

Let's take a quick look at the matrix itself:

To render this image in our Python notebook, we can make use of matplotlib, but this time instead of using this library to display our plots, we'll use it to display our images. To do this, we'll make use of the <code>plt.imshow()</code> function. <a href="https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html" rel="noopener noreferrer" target="_blank"><u>Documentation for <code>plt.imshow()</code> can be found here.</u></a> 

If we dig into the documentation, we'll see that we can pass a 2D numpy array to the function along with additional arguments to adjust how the image will be displayed/rendered. First, let's display our image (the above matrix of zeros) with the default parameters.

You can see that by default, the colormap <code>cmap</code> will be a colormap called <a href="https://cran.r-project.org/web/packages/viridis/vignettes/intro-to-viridis.html" rel="noopener noreferrer"><u>'viridis', which was developed to improve readability and accessibility of quantitative data visualizations, such as as heatmaps</u></a>. To display our image as a grayscale image, we can pass the string <code>'gray'</code> to the <code>cmap</code> parameter.

You can see that our matrix of zeros is rendered as just a pure black image. What if we change all the values of our 2D matrix to 255 and then display it using <code>plt.imshow()</code>?

Recall that we can perform math operations on a matrix.

To then properly display our image, we need to pass additional arguments to <code>plt.imshow()</code>. Specifically, we'll specify a <code>vmin</code> and <code>vmax</code>, which specify how values for pure black and pure white will be assigned, respectively.

```python
plt.imshow(our_image, cmap='gray', vmin=0, vmax=255)
```

That way, it'll render the image as an 8-bit grayscale image.

You should see that the image renders as just pure white, so as we went over earlier, in an 8-bit grayscale image, the value of <code>0</code> indicates pure black and the value of <code>255</code> indicates pure white.

Let's then take a look at the values in between by generating a 2D matrix of random integers using <a href="https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.integers.html" rel="noopener noreferrer"><u>numpy's random generator for integers.</u></a>

The general set up is below:

```python
rng = np.random.default_rng()
```
Which initializes the bitgenerator, and we can generate random integers using:

```python
our_image = rng.integers(low, high, size)
```
Where we pass values to <code>low</code>, <code>high</code>, and <code>size</code> to generate our 10x10 2D matrix. <a href="https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.integers.html" rel="noopener noreferrer"><u>Documentation for <code>rng.integers()</code> can be found here.</a></u>

Then see how the image looks like using <code>plt.imshow()</code>, while specifying the <code>cmap</code> and <code>vmin</code> and <code>vmax</code> like we did before.

Take a quick look at the matrix itself to see how the gray pixels are represented

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #2: Exploring a real grayscale image</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 800px;"></hr>

To make things more interesting, let's import a real 8-bit grayscale image to render it in our notebook. In this case, rather than using the panadas package and <code>pd.read_csv()</code> to import our file, we'll be using matplotlib's function <code>plt.imread()</code>. This function will read the data contained within an image file into a multidimensional array that we can then work with in Python. Like importing a file with <code>pd.read_csv()</code>, all we need to do is pass our file path as a string to <code>plt.imread()</code> so that it can read in our image file.

Let's give it a try with a grayscale image of Liebchen located in the following file path:

```python
'./ref-images/liebchen-gray.jpg'
```

Then let's go ahead an display the image in our notebook using <code>plt.imshow()</code>.

If you want to hide the axes, you can call up the axes from the plot and set it to <code>False</code>.

```python
plt.axis(False)
```

Like with our example before, you can see that an 8-bit grayscale image is represented as a 2D matrix containing values from 0 (black) to 255 (white). We can take a look at a 10x10 slice of our imported image to see how the image file is understood by Python.

What you can see is that like our earlier example with the matrices we generated, each pixel of the image we imported is represented by a value that represents its grayscale value. In this case, our image file is an 8-bit grayscale image, so we have a single 2D array where the values can span 0-255. 

Recall from earlier that we can also specify what value corresponds to pure black and what value corresponds to pure white using the <code>vmin</code> and <code>vmax</code> parameters, respectively. You'll want to keep in mind that the underlying data isn't changed by adjusting the <code>vmin</code> and the <code>vmax</code>. These parameters just change how the image is displayed/rendered by changing how the values are assigned grayscale shades.

Give it a try below by changing <code>vmin</code> and <code>vmax</code> for our image of Liebchen.

Since our images are just 2D arrays, we can use slice notation to pull out portions of our image and then display that slice using <code>plt.imshow()</code>.

<h2>Thresholding an image</h2>

Let's use our image of Liebchen as an example for thresholding an image and understanding what's going on under the hood. When working with a single channel (like our grayscale image), thresholding essentially converts the image's bit-depth to a 1-bit (binary) image. Since the value space of our Liebchen image is 256 possible values, which will need to be collapsed into just 2 possible values, we need to specify how we are going to bin (or assign) the 256 values to either 0 or 1. 

If you recall that the booleans <code>False</code> and <code>True</code> can also be represented by the values <code>0</code> and <code>1</code>, the operation to convert our 8-bit image to a 1-bit binary image looks similar to a conditional statement, where we can specify a value to use as a threshold or cutoff for assigning 0 and 1.

See if you can take what you know about conditional statements to then threshold the 8-bit image of Liebchen by applying a conditional statement to the 2D matrix and assign the resulting matrix to a new variable.

How does the resulting matrix look like? And how does the image render using <code>plt.imshow()</code>?

You should be able to see now that your thresholded array contains booleans. This is because we applied our conditional statement to each element in our 2D arrays, resulting in a boolean output corresponding to whether or not each element in the array met that condition. And the rendered image is now just a binary image.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #3: What about color images?</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 700px;"></hr>

So far, we've just been playing with grayscale images, which are a single 2D matrix. True-color images on a computer are represented by three channels that are overlaid one on top of the other to display the final color. So what this means is that instead of a single 2D matrix, a true-color image contains three 2D matrices, where each 2D matrix represents a specific channel, either red, green, or blue.

Let's use our random integer generator <code>rng.intergers()</code> to create a 3D matrix of random integers by providing it with a tuple with three positions. For a 10x10, 3-channel image, we'll provide it with a tuple that looks like <code>(10, 10, 3)</code>. And if we want an 8-bit image, we can then limit the range to be from 0-255 (inclusive).

Generate your 10x10x3 array below using <code>rng.integers()</code>, similarly to how we did it for our grayscale example.

Then pass your matrix to <code>plt.imshow()</code>.

You should see now that the image is displayed as a true-color image rather than a grayscale image.

<h2>Pull out specific channels from a true-color image</h2>

If we wanted to pull information from a specific channel, we can use slice notation to pull out either the red, green or blue channel, where the channels are specified by the third axis in our 3D array.

```python
red_channel = color_img[:, :, 0]
green_channel = color_img[:, :, 1]
blue_channel = color_img[;, ;, 2]
```

What this means is that if we look only along the third axis, we have 3 elements that each are 2D matrices: 

<code>[red_channel, green_channel, blue_channel]</code>

You'll want to keep this order (RGB) in mind for later when you are combining channels together or trying to pseudocolor a single channel because the order in which the 2D arrays are arranged along the third axis (channel axis) dictates their displayed color.

See if you can pull out each channel and display them individually below.

Let's take a look at each individual matrix object's values as well.

You should be able to see that for our 8-bit true-color image, all the individual channels are still represented by values ranging from 0-255.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #4: Exploring color images</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 700px;"></hr>

Now let's see how a real true-color image file is understood by the computer by playing around with a color image in our notebooks. Like with our grayscale image of Liebchen, let's import a true-color image of Liebchen located in the following file path:

```python
'./ref-images/liebchen-color.jpg'
```

Let's then display the imported true-color image of Liebchen.

Let's then take a look at what's going on underneath the hood to understand what information the image holds. How does the object itself look like if you just use <code>print()</code>?

What you can see is that based on the syntax of our 3D matrix, it is a bit difficult to exactly tease out complete information on each channel by simply looking at the object in its entirety. 

We know from our experience with 2D objects, like pandas DataFrames, that the dimensions are ordered as <code>&lbrack;a, b&rbrack;</code> with the first dimension <code>a</code> corresponding to the number of rows (height of our image), and the second dimension <code>b</code> to the number of columns (width of our image). 

So then our 3D image is just our 2D arrays stacked one on top of another along a third dimension <code>c</code>:

<p style="font-size: 16px; text-align: center;"><strong>True-color image as 3D array</strong></p>
<img src="./ref-images/array-explanation.jpg" style="height: 300px; margin: auto"/>

Our true-color image can then be understood as being represented as the 3D array <code>&lbrack;a, b, c&rbrack;</code>.

This means that for a 3-channel true-color image, the values for <code>c</code> will be either <code>0</code>, <code>1</code>, or <code>2</code>, with each value of <code>c</code> containing the 2D matrix of intensity values ranging from 0-255. These values simply represent intensity of light as a grayscale but then are assigned intensities for the colors red, green, and blue based on their position in the third axis. Since computer screens follow the additive color model (see below), each channel will "add" to the other channels resulting in the color displayed. So if all three channels have their maximum value, then the color displayed is pure white.

<p style="font-size: 16px; text-align: center;"><strong>Additive color model</strong></p>
<img src="./ref-images/additive-color.jpg" style="height: 300px; margin: auto"/></div>

Let's pull out a single row from our true-color image to look at the RGB values.

And if we wanted all three channels of a single pixel:

So if we wanted a single channel, we can specify that as the third dimension. For a single pixel:

If we wanted the full 2D array for a single channel, we'll need to specify that we want all values from the first two dimensions, and just a single value for our third dimension.

We can then plot a single channel using the same function we used before.

Let's plot all three channels in grayscale side-by-side:

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #5: Assembling a composite image</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 850px;"></hr>

More often than not, we're imaging our samples one channel at a time, so if we want to display our images as a composite, we need to construct a 3D array out of our separate 2D arrays. We can do this by using the <code>np.dstack()</code> function, which will stack/concatenate our arrays along the third axis. <a href="https://numpy.org/doc/stable/reference/generated/numpy.dstack.html" rel="noopener noreferrer"><u>Documentation for <code>np.dstack()</code> can be found here.</u></a>

For this example, let's split the true-color Liebchen image into its three individual channels, so that each channel is held by its own variable.

Then use <code>np.dstack()</code> to bring all the channels back together as a single 3D matrix to display again using <code>plt.imshow()</code>.

Recall from earlier that the position of our 2D arrays in the 3rd dimension, specifies what channel it's in, so it's important to keep this in mind so that you know what channel(s) you are pulling out or compositing.

For example, if we mix up the channels, our true-color image won't be accurate to what we want it to show. Give it a try below to see what happens if you mix up channels.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #6: Pseudocolor an image</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 700px;"></hr>

We can take what we know about how true-color images are built up as a composite of three channels as well as what we know about the additive color model to also pseudocolor our images. 

See what happens if we repeat a single channel in all three channels. For this example, take the red channel and then use <code>np.dstack()</code> stack together the red channel three times. How does the resulting image look like?

You should then see that the red channel is simply displayed as what appears to be a grayscale image.

<h2>Specify a pseudocolor</h2>

We can then take what we know about mathematical operations on arrays to then adjust the values for each channel. If we want to then pseudocolor our grayscale image red, we can zero-out the green and blue channels.

We can also achieve the same result by setting up a 2D array of zeros of the same dimensions as our image, and placing it in the positions corresponding to the channels we don't want to use. To set up an array of zeros that are the same dimensions as another object, we can make use of the <code>np.zeros_like()</code> function. <a href="https://numpy.org/doc/stable/reference/generated/numpy.zeros_like.html" rel="noopener noreferrer"><u>Documentation for <code>np.zeros_like()</code> can be found here.</u></a>

This function takes an object as an argument and then will initialize an array of zeros that matches the shape of the object that it was given, so if we provide it with one of our 2D arrays corresponding to our image, it will generate an array of zeros of the same shape as our image.

Set up an array of zeros shaped like the Liebchen true-color image below.

Then assemble a three channel image to display with <code>plt.imshow()</code>, where the green and blue channels are assigned our array of zeros and generate an image, and the red channel is assigned our red channel that we pulled our from our true-color image.

We can also mess with the channels and values to pseudocolor our image in a color that isn't exactly red, green, or blue. See what happens if you set two channels to be red instead of just one, and leave the remaining channel as a matrix of zeros.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise # 7: Exporting processed images</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 750px;"></hr>

Since we're using matplotlib to render our images, we can make use of our usual <code>plt.savefig()</code> or <code>fig.savefig()</code> functions to export our processed images. This time, instead of specifying our images with the extension <code>.pdf</code>, we can specify <code>.jpg</code> or another image extension, and our image will be exported as that file type.

For this example, take one of the images that you've rendered in your notebook and then export it using <code>plt.savefig()</code>.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #8: Explore fluorescence data</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 850px;"></hr>

Now, let's take what we've learned and then apply it to fluorescence data from MCB201A. First we'll import our fluorescence data files under the <code>data</code> subdirectory. We'll import all the files and apply what we know about how images are represented to process our fluorescence data and analyze them later on to extract quantitative data.

You can load it in one by one or if you remember from when we worked with multiple files for statistical analysis, see if you can apply that here to load in the data for today.

<h2>Pseudocolor your fluorescence images</h2>

See if you can take what you've learned to then pseudocolor an individual channel from our set of images. Select either the no serum or serum stimulation condition and psuedocolor each channel for that treatment condition.

<h2>Render a three-channel composite image</h2>

In ImageJ, we can pull all three channels together to render a composite image, and similarly, we can do the same for our fluorescence images using Python.

Take what you learned earlier and pull all three channels for a single image together to render a composite of that image. If you run into issues rendering the image, consider what mathematical operations you can perform to have your image display more closely to what you anticipate from your experiences in MCB201A.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise #9: Threshold an image to segment nuclei</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 950px;"></hr>

For this exercise, we'll begin processing our image to analyze them much like how we analyzed them in MCB201A using ImageJ. First, we'll take a look at the underlying values of our DAPI images to identify what value we can set as a potentially "good" threshold to segment our nuclei from our background. 

To identify a potentially "good" threshold, take a look at the distribution of all pixel intensities for our DAPI images by using seaborn's <code>sns.histplot()</code> to visualize a histogram of pixel intensities. <a href="https://seaborn.pydata.org/generated/seaborn.histplot.html" rel="noopener noreferrer"><u>Documentation for <code>sns.histplot()</code> can be found here.</u></a>

Since our DAPI images are a 2D array, it won't plot neatly if we pass it to <code>sns.histplot()</code> as a 2D array. You'll need to flatten the array by using <code>np.ndarray.flatten()</code> to turn your 2D array into a 1D array, so that it can be used to plot a simple histogram.

Use the code cell below to plot a histogram displaying the distribution of values of our two DAPI channels (serum and no serum images), and plot each channel as its own distribution on a single plot.

You can set the <code>bins</code> parameter in <code>sns.histplot()</code> to <code>50</code> for visualizing the pixel intensity distribution of both DAPI images. You may need to adjust the y-axis limits to be able to see the counts for higher gray values.

Taking a look at the distribution of pixel intensity values, what would a "good" value to use as a threshold to segment our image between background and nuclei? In other words, by looking at the shape of our histogram, how can we decide where a potential breakpoint in our image is located that will allow us to be confident the pixel values below that point are "background" and the pixel values above that point are nuclei? 

Plot a single vertical line on your histogram corresponding to your breakpoint.

Now that you've identified a "good" spot to set your threshold, use a conditional statement to threshold your DAPI images, thereby converting it into binary images.

Take a look at the resulting 2D array object: 

Now render your binary images side by side to see how the nuclei have been segmented for both images.

<h1 style="font-size: 40px; margin-bottom: 0px;">Guided: Fill holes in a thresholded image</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 800px;"></hr>

There are also many other ways to calculate thresholds, and when we reconvene, we'll go over additional ways to threshold our images and then continue processing our segmented nuclei.

For those of you working ahead, take a look at <a href="https://scikit-image.org/docs/stable/api/skimage.filters.html#skimage.filters.try_all_threshold" rel="noopener noreferrer"><u>the documentation example for <code>ski.filters.try_all_threshold()</code></u></a> to see if you can set this up for a single DAPI channel image, such as the no serum condition's DAPI channel.

If we then wanted <a href="https://scikit-image.org/docs/stable/api/skimage.filters.html" rel="noopener noreferrer"><u>to use one of these algorithms to threshold our nuclei</u></a>, we can call them up from the <code>scikit-image</code> package.

For example, let's say we like the Otsu algorithm, then we can pass our image to the <code>ski.filters.threshold_otsu()</code> function.

```python
otsu_tresh = ski.filters.threshold_otsu(image)
```

This function will then return a single value because it is calculating what a "good" breakpoint is using the Otsu algorithm. If we preferred another algorithm, we could use one of the other available functions as well.

Give the Otsu algorithm a try below to threshold your DAPI images.

Sometimes our thresholded image isn't exactly what we want it to be, and in the case of a few of our nuclei, we have some holes that will be helpful to fill so that we capture the entirety of each nuclear region. To do this, we can make use of the <code>ndi.fill_binary_holes()</code> function, which is a quick way to fill in holes in our multidimensional binary array. <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.binary_fill_holes.html" rel="noopener noreferrer"><u>Documentation for <code>ndi.fill_binary_holes()</code> can be found here.</u></a>

First take a look at the thresholded image for your no serum DAPI channel image.

We can see that there are holes within our segmented nuclei, and we can fill them in by passing the binary image to the <code>ndi.binary_fill_holes()</code> function, which will then return a new 2D array.

Another way, would be to do a dilation followed by an erosion, which will help to fill in gaps while limiting how much we expand our region of interest. To do this, we'll make use of some functions within scikit-image:

<ul>
    <li><code>ski.morphology.disk()</code> - this creates a binary circle for us to use as a footprint in our dilation and erosion. <a href="https://scikit-image.org/docs/stable/api/skimage.morphology.html#skimage.morphology.disk" rel="noopener noreferrer"><u>Documentation is here.</u></a></li>
    <li><code>ski.morphology.dilation()</code> - this will increase the size of bright regions (our ROI). <a href="https://scikit-image.org/docs/stable/auto_examples/applications/plot_morphology.html#dilation" rel="noopener noreferrer"><u>Documentation is here.</u></a></li>
    <li><code>ski.morphology.erosion()</code> - this will increase the size of dark regions (our background). <a href="https://scikit-image.org/docs/stable/auto_examples/applications/plot_morphology.html#erosion" rel="noopener noreferrer"><u>Documentation is here.</u></a></li>
    <li><code>ski.morphology.closing()</code> - this will performs a dilation followed by an erosion, and can be another way to achieve a similar result. <a href="https://scikit-image.org/docs/stable/auto_examples/applications/plot_morphology.html#closing" rel="noopener noreferrer"><u>Documentation is here.</u></a></li>
</ul>

First dilate the binary image to fill holes:

Then erode the edges away to shrink the ROI back to roughly where it was before:

<h1 style="font-size: 40px; margin-bottom: 0px;">Guided: Use binary threshold as a mask</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 850px;"></hr>

Recall back to MCB201A, where we wanted to measure the signal intensity within the nucleus for our fluorescence images. What we'll do here is make use of what we now know about working with images and mathematical operations on arrays to use our segmented nuclei as a mask in order to look at just our region of interest (ROI) in the green and red channels.

For those of you working ahead, recall that our thresholded image contains booleans, which can be represented as <code>0</code> for <code>False</code> and <code>1</code> for <code>True</code>. Given what you know about multiplication, how can you use your thresholded image as a mask on your YAP channel image to just look at the nuclei signal intensity?

<h1 style="font-size: 40px; margin-bottom: 0px;">Challenge #1: Segment cell cytoplasm</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 750px;"></hr>

For this challenge, practice applying what you've learned so far to image analysis to pull cells as our ROI. Like we did in MCB201A, you can use the F-actin stain to segment just the cytoplasm.

See if you can try all the thresholds for the no serum condition's F-actin stain to see what threshold algorithm works well in this case.

Pick one that looks good to you and then segment your F-actin image.

Does your threshold algorithm leave holes that you need to fill?

Take a look at your final thresholded image using <code>plt.imshow()</code>.

<h1 style="font-size: 40px; margin-bottom: 0px;">Challenge #2: Remove nuclei from binary</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 950px;"></hr>

We now have binaries for our nuclei and our cell cytoplasm, where the cell cytoplasm threshold also contains the area occupied by the nucleus. Recall from MCB201A that we deselected our nuclei to get just the signal from the cytoplasm. See if you can make use of our two binary images to remove the nuclear region from our cell cytoplasm binary.

Now use your cytoplasm only binary as a mask to look at just the cytoplasmic YAP signal.

And compare that side-by-side to the nuclear YAP image: